# 🔥 Training Model Sensor Deteksi Kebakaran (Multi-Class, PPM)

| Item | Detail |
|------|--------|
| Dataset | `dataset_sensor.csv` (RAW ADC dari ESP32) |
| Konversi | RAW ADC → PPM murni dari kurva datasheet |
| Fitur | mq4_ppm, mq5_ppm, mq135_ppm, mq2_ppm, mq7_ppm, mq3_ppm |
| Target | Multi-class: Clean=0, Gasoline=1, Mixture=2, Smoke=3 |
| Output | `fire_detection_rf.pkl` + `scaler.pkl` |

> **Pipeline:** RAW ADC → PPM (tanpa kalibrasi manual) → Scaler → Model
> 
> Tidak ada baseline subtraction. Model mengklasifikasi murni dari pola PPM.

## 1. Install & Import

In [ ]:
!pip install -q scikit-learn xgboost pandas matplotlib seaborn

import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier

print('✅ Import selesai!')

## 2. Load Dataset & Konversi PPM

In [ ]:
DATASET_PATHS = ['dataset_sensor.csv', '/kaggle/input/dataset-sensor/dataset_sensor.csv', '/kaggle/input/datasets/jeeerzr/mq-sensor-dataset/dataset_sensor.csv', '/content/dataset_sensor.csv']
df = None
for p in DATASET_PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f'✅ Dataset loaded: {p}')
        break
assert df is not None, '❌ Dataset tidak ditemukan!'

RAW_COLS = ['mq4', 'mq5', 'mq135', 'mq2', 'mq7', 'mq3']
PPM_COLS = ['mq4_ppm', 'mq5_ppm', 'mq135_ppm', 'mq2_ppm', 'mq7_ppm', 'mq3_ppm']

print(f'Shape: {df.shape}')
print(df['label'].value_counts())

In [ ]:
# ==============================================================
# Konversi RAW ADC → PPM (IDENTIK dengan ai_engine.py!)
# Tanpa kalibrasi manual, tanpa baseline subtraction.
# PPM = a * (voltage_ratio)^b
# ==============================================================
VCC_SENSOR = 5.0
VCC_ADC = 3.3
ADC_MAX = 4095.0

SENSOR_CURVES = {
    'mq4':  {'a': 1012.7, 'b': -2.786},
    'mq7':  {'a': 99.042, 'b': -1.518},
    'mq5':  {'a': 1000.5, 'b': -2.186},
    'mq135': {'a': 110.47, 'b': -2.862},
    'mq2':  {'a': 574.25, 'b': -2.222},
    'mq3':  {'a': 0.3934, 'b': -1.504},
}

SENSOR_NAMES = {
    'mq4': 'MQ-4 (Metana)', 'mq5': 'MQ-5 (LPG)',
    'mq135': 'MQ-135 (Air Quality)', 'mq2': 'MQ-2 (Smoke/Gas)',
    'mq7': 'MQ-7 (CO)', 'mq3': 'MQ-3 (Alkohol)',
}

def raw_adc_to_ppm(sensor_key, raw_adc):
    """RAW ADC → PPM murni. Sama persis dengan ai_engine.py."""
    if sensor_key not in SENSOR_CURVES:
        return 0.0
    raw_adc = np.clip(raw_adc, 1, ADC_MAX - 1)
    vout = (raw_adc / ADC_MAX) * VCC_ADC
    voltage_ratio = (VCC_SENSOR - vout) / vout
    a, b = SENSOR_CURVES[sensor_key]['a'], SENSOR_CURVES[sensor_key]['b']
    try:
        ppm = a * math.pow(voltage_ratio, b)
    except:
        ppm = 0.0
    return max(0.0, ppm)

# Konversi kolom
for raw_col, ppm_col in zip(RAW_COLS, PPM_COLS):
    df[ppm_col] = df[raw_col].apply(lambda x, k=raw_col: raw_adc_to_ppm(k, x))

print('\n✅ Konversi RAW → PPM selesai!')
print('\nStatistik PPM per label:')
print(df.groupby('label')[PPM_COLS].mean().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribusi PPM per Sensor & Label', fontsize=16, fontweight='bold')
for i, (raw_col, ppm_col) in enumerate(zip(RAW_COLS, PPM_COLS)):
    ax = axes[i // 3][i % 3]
    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label][ppm_col]
        ax.hist(subset, bins=50, alpha=0.5, label=label, density=True)
    ax.set_title(SENSOR_NAMES[raw_col] + ' (PPM)', fontweight='bold')
    ax.set_xlabel('PPM')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Preprocessing & Split

In [ ]:
le = LabelEncoder()
le.fit(sorted(df['label'].unique()))
df['target'] = le.transform(df['label'])

CLASS_NAMES = list(le.classes_)
print('Mapping kelas:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i} = {name} ({(df["target"] == i).sum()} samples)')

X = df[PPM_COLS].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Fitur: {PPM_COLS}')

## 4. Training: Random Forest vs XGBoost

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('RANDOM FOREST')
rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
rf_cv = cross_val_score(rf, X_train_scaled, y_train, cv=cv, scoring='accuracy')
rf_pred = rf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)
print(f'CV: {rf_cv.mean():.4f} | Test: {rf_acc:.4f}')
print(classification_report(y_test, rf_pred, target_names=CLASS_NAMES))

print('XGBOOST')
xgb = XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='mlogloss', n_jobs=-1)
xgb.fit(X_train_scaled, y_train)
xgb_cv = cross_val_score(xgb, X_train_scaled, y_train, cv=cv, scoring='accuracy')
xgb_pred = xgb.predict(X_test_scaled)
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f'CV: {xgb_cv.mean():.4f} | Test: {xgb_acc:.4f}')
print(classification_report(y_test, xgb_pred, target_names=CLASS_NAMES))

if xgb_acc >= rf_acc:
    best_model, best_name, best_pred = xgb, 'XGBoost', xgb_pred
else:
    best_model, best_name, best_pred = rf, 'Random Forest', rf_pred
print(f'\n🏆 Model terpilih: {best_name}')

## 5. Evaluasi & Simulasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title(f'Confusion Matrix ({best_name})', fontweight='bold')
axes[0].set_ylabel('Aktual'); axes[0].set_xlabel('Prediksi')
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
    labels = [SENSOR_NAMES[c] for c in RAW_COLS]
    idx = np.argsort(imp)
    axes[1].barh(range(len(idx)), imp[idx], color='steelblue')
    axes[1].set_yticks(range(len(idx))); axes[1].set_yticklabels([labels[i] for i in idx])
    axes[1].set_title('Feature Importance (PPM)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nSIMULASI PREDIKSI (rata-rata PPM per label):')
for label in sorted(df['label'].unique()):
    sample = df[df['label'] == label][PPM_COLS].mean().values.reshape(1, -1)
    proba = best_model.predict_proba(scaler.transform(sample))[0]
    pred = CLASS_NAMES[np.argmax(proba)]
    danger = 100 - proba[CLASS_NAMES.index('Clean')] * 100 if 'Clean' in CLASS_NAMES else 0
    probs = ' | '.join([f'{c}: {p*100:.1f}%' for c, p in zip(CLASS_NAMES, proba)])
    print(f'{label:12s} → {pred:10s} | Danger: {danger:5.1f}% | {probs}')

## 6. Export Model

In [ ]:
MODEL_OUT = 'fire_detection_rf.pkl'
SCALER_OUT = 'scaler.pkl'
scaler.feature_names_in_ = np.array(PPM_COLS)
joblib.dump(best_model, MODEL_OUT)
joblib.dump(scaler, SCALER_OUT)

print(f'✅ Model: {MODEL_OUT} ({os.path.getsize(MODEL_OUT)/1024:.1f} KB)')
print(f'✅ Scaler: {SCALER_OUT} ({os.path.getsize(SCALER_OUT)/1024:.1f} KB)')
print(f'\n📌 Kelas: {CLASS_NAMES}')
print(f'📌 Fitur: {PPM_COLS}')
print(f'\n⚠️ Rumus PPM = a * (voltage_ratio)^b')
print(f'⚠️ Tanpa kalibrasi manual / baseline subtraction')

try:
    from google.colab import files
    files.download(MODEL_OUT); files.download(SCALER_OUT)
except:
    print('\n💾 Kaggle: Klik Output tab untuk download.')